# EDA inicial sobre los datos crudos (`data/raw/`)

Este notebook es retroactivo: documenta los problemas de calidad que llevaron a las
decisiones de limpieza ya implementadas en `etl/transform/` y `etl/load/schema_mapping.py`
(sentinels, nulos intencionales, columnas mal etiquetadas). Cada hallazgo termina con un
enlace al archivo del ETL que lo resuelve, para que la limpieza quede trazada al análisis
que la motivó.

In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

archivos = sorted(RAW.glob("*.csv"))
dfs = {f.stem: pd.read_csv(f) for f in archivos}

print(f"{len(dfs)} archivos cargados:")
for nombre in dfs:
    print(" -", nombre)

## 1. Resumen general: forma, tipos, nulos y duplicados por tabla

In [ ]:
resumen = []
for nombre, df in dfs.items():
    resumen.append({
        "tabla": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
        "nulos_totales": int(df.isna().sum().sum()),
        "columnas_con_nulos": list(df.columns[df.isna().any()]),
        "duplicados_filas_completas": int(df.duplicated().sum()),
    })

resumen_df = pd.DataFrame(resumen)
resumen_df

La columna `columnas_con_nulos` es la pista principal: cualquier tabla que aparezca ahí
tiene datos faltantes que hay que decidir cómo tratar antes de cargarlos a Postgres.

## 2. Hallazgo: `estado` nulo en `usuarios` y `productos`

Ambas tablas tienen una columna `estado` con valores `NaN` — no es un error de lectura,
el seeder los genera a propósito para simular datos sucios.

In [ ]:
print("usuarios.estado:")
print(dfs["usuarios"]["estado"].value_counts(dropna=False))
print()
print("productos.estado:")
print(dfs["productos"]["estado"].value_counts(dropna=False))

**Decisión tomada:** en vez de descartar esas filas (son ~25% de los registros), se definió
un conjunto de valores válidos por campo (`etl/transform/enums.py`) y cualquier valor fuera de
ese conjunto — incluyendo `NaN` — se normaliza a `"Desconocido"` con `normalize_categorical`
(`src/core/errors/handlers.py`), aplicado en `clean_usuarios` y `clean_productos`.

## 3. Hallazgo: sentinel `0` en `categorias.id_categoria_padre`

Las categorías raíz (sin padre) no quedan con un valor nulo, sino con `0` — un id que no
existe en la propia tabla.

In [ ]:
categorias = dfs["categorias"]
print(categorias["id_categoria_padre"].value_counts(dropna=False).head())
print()
print("¿Existe id_categoria == 0?", (categorias["id_categoria"] == 0).any())

`id_categoria` nunca es `0`, así que cualquier fila con `id_categoria_padre == 0` en realidad
no tiene padre — es el mismo significado que `NaN`, solo que mal codificado.

**Decisión tomada:** en `etl/transform/categorias_transform.py`, todo `id_categoria_padre == 0`
se convierte explícitamente a `NaN` antes de validar integridad referencial, para no
interpretarlo como una referencia real.

## 4. Hallazgo: sentinels en `envios.direccion` (envíos sin dirección válida)

Cuando un cliente no tiene ninguna dirección registrada, el seeder no deja el campo vacío
de una forma consistente: usa una mezcla de `0`, `None` y strings de solo espacios.

In [ ]:
envios = dfs["envios"]
print(envios["direccion"].apply(lambda v: v if str(v).strip() not in ("", "0") and pd.notna(v) else "<sentinel>").value_counts().tail(10))

sentinels = envios["direccion"].isin([0, "0"]) | envios["direccion"].isna() | envios["direccion"].astype(str).str.strip().eq("")
print()
print("Filas con sentinel en 'direccion':", sentinels.sum(), "de", len(envios))

**Decisión tomada:** en `etl/load/schema_mapping.py`, la columna se castea a numérico y tanto
`0` como los valores no numéricos (`None`, espacios) se convierten a `NaN`; las filas sin
`id_direccion` válido se descartan antes de cargar, porque no hay forma de inventar una
dirección real para esos envíos.

## 5. Hallazgo: columnas mal etiquetadas en `direcciones`

El nombre de columna `"pais ciudad"` (con espacio, una sola columna) ya es una señal de que
algo está mal. Comparando los valores reales contra lo que deberían contener:

In [ ]:
direcciones = dfs["direcciones"]
print(direcciones[["pais ciudad", "departamento"]].head(8))
print()
print("Valores únicos en 'pais ciudad':", direcciones["pais ciudad"].unique())

`"pais ciudad"` siempre vale `"Colombia"` — es en realidad el país. Y `"departamento"`
contiene nombres de ciudad (`fake.city()`), no departamentos — el seeder generó los datos en
el orden incorrecto respecto a las columnas declaradas. No existe ningún dato real de
departamento en el CSV.

**Decisión tomada:** en `etl/load/schema_mapping.py` se renombra `"pais ciudad"` → `pais` y
`departamento` → `ciudad` (recuperando el valor real), y como el departamento verdadero nunca
se generó, se completa con un valor fijo `"Sin dato"` en vez de bloquear la carga.

## 6. Mapa hallazgo → limpieza

| Hallazgo | Tabla | Dónde se resuelve |
|---|---|---|
| `estado` nulo (dato sucio intencional) | `usuarios`, `productos` | `etl/transform/enums.py` + `normalize_categorical` |
| Sentinel `0` como "sin padre" | `categorias` | `etl/transform/categorias_transform.py` |
| Sentinels mixtos como "sin dirección" | `envios` | `etl/load/schema_mapping.py` |
| Columnas mal etiquetadas / dato faltante | `direcciones` | `etl/load/schema_mapping.py` |

Estos cuatro hallazgos son los únicos problemas de datos "basura" inyectados a propósito en
todo el dataset (confirmado revisando los 18 seeders); el resto de la limpieza (tipos, fechas,
duplicados) es mantenimiento estándar ya cubierto en cada `*_transform.py`.